
# Data Source
[See OpenAddresses project.](https://batch.openaddresses.io/data#map=0/0/0)

In this notebook, we used buildings and addresses from `au/vic/statewide` in this example but this analysis can be applied globally to your own datasets as required.

Other Datasets you can use to explore, but not covered in this notebook:

* Overture Maps, available on the Marketplace to consume via Delta Sharing. It covers: Addresses, Buildings, Places, Divisions, Transportation, as well as base themes


## Environment setup

* Make sure you are using DBR 17.1+, or minimum of Environment version 4 on SVLS compute.

In [0]:
%sql

USE CATALOG geo;
USE SCHEMA openaddresses;

In [0]:
# Load VIC Buildings
bld = spark.read.format("json").load("/Volumes/geo/openaddresses/address_ref/vic buildings - state.geojson")
bld.write.format("delta").mode("overwrite").saveAsTable("geo.openaddresses.bronze_vic_buildings")

In [0]:
# TODO Load VIC Addresses
addr = spark.read.format("json").load("/Volumes/geo/openaddresses/address_ref/vic addresses - state.geojson")
addr.write.format("delta").mode("overwrite").saveAsTable("geo.openaddresses.bronze_vic_addresses")

In [0]:
%sql
-- Parse GeoJSON in CRS 4326 to WKT
CREATE OR REPLACE TABLE geo.openaddresses.silver_vic_buildings AS
SELECT
  'building' AS object_type,
  'VIC' AS state,
  'Australia' AS country,
  st_astext(st_geomfromgeojson(to_json(geometry))) as geom_4326 -- Handle GeoJSON in CRS 4326, no transformation required. Store as WKT
FROM geo.openaddresses.bronze_vic_buildings;

In [0]:
%sql
-- Parse GeoJSON in CRS 4326 to WKT
CREATE OR REPLACE TABLE geo.openaddresses.silver_vic_addresses AS
SELECT
  properties.id::STRING as property_id,
  initcap(concat(properties.number::STRING, ' ', properties.street::STRING)) as street_address,
  initcap(properties.city::STRING) as city,
  properties.region::STRING as state,
  properties.postcode::STRING as postcode,
  'address' AS object_type,
  'Australia' AS country,
  type,
  st_astext(st_geomfromgeojson(to_json(geometry))) as geom_4326 -- Handle GeoJSON in CRS 4326, no transformation required. Store as WKT
FROM geo.openaddresses.bronze_vic_addresses
-- TABLESAMPLE (100 ROWS);

In [0]:
%sql
-- SELECT * FROM geo.openaddresses.silver_vic_buildings;
SELECT * FROM geo.openaddresses.silver_vic_addresses;

In [0]:
%pip install folium shapely
%restart_python

In [0]:
%sql
SELECT COUNT(*) FROM geo.openaddresses.silver_vic_addresses;
-- Total of 4M fake addresses in VIC

In [0]:
%sql
SELECT COUNT(*) FROM geo.openaddresses.silver_vic_buildings;
-- Total of 32K fake buildings in VIC

In [0]:
# Visualise Customer Addresses (Fake Data) as points

import folium
from shapely import wkt

m = folium.Map(location=[-37.81, 144.96], zoom_start=8, tiles='CartoDB positron') # Use an alternative to OpenStreetMap

df_geom = spark.sql("SELECT * FROM geo.openaddresses.silver_vic_addresses TABLESAMPLE (1000 ROWS)").toPandas()
for _, row in df_geom.iterrows():
    folium.GeoJson(wkt.loads(row['geom_4326'])).add_to(m)

# Display the map
display(m)


In [0]:
# Visualise Buildings (Fake Data) as polygons

import folium
from shapely import wkt

m = folium.Map(location=[-37.81, 144.96], zoom_start=14, tiles='CartoDB positron') # Use an alternative to OpenStreetMap

# Add buildings and addresses to the map (1K rows ea)
df_geom = spark.sql("SELECT * FROM geo.openaddresses.silver_vic_buildings TABLESAMPLE (1000 ROWS)").toPandas()
for _, row in df_geom.iterrows():
    folium.GeoJson(wkt.loads(row['geom_4326'])).add_to(m)

# Display the map
display(m)
